# Лабораторная работа №1

## Выполнил студент группы 6401-010302D Афанасьев Всеволод

## Чтение данных и первая обработка

In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf

import itertools
from math import radians, sin, cos, sqrt, atan2

conf = SparkConf().setAppName("Lab1_PySpark").setMaster("local[*]")

sc = SparkContext.getOrCreate(conf=conf)

spark = SparkSession.builder \
    .appName("Lab1_PySpark") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {sc.version}")
print(f"Python version: {sc.pythonVer}")

Spark version: 3.3.0
Python version: 3.9


In [2]:
tripData = sc.textFile("trip.csv")
tripsHeader = tripData.first()
trips = tripData.filter(lambda row: row != tripsHeader).map(lambda row: row.split(",", -1))

stationData = sc.textFile("station.csv")
stationsHeader = stationData.first()
stations = stationData.filter(lambda row: row != stationsHeader).map(lambda row: row.split(",", -1))

In [3]:
list(enumerate(tripsHeader.split(",")))

[(0, 'id'),
 (1, 'duration'),
 (2, 'start_date'),
 (3, 'start_station_name'),
 (4, 'start_station_id'),
 (5, 'end_date'),
 (6, 'end_station_name'),
 (7, 'end_station_id'),
 (8, 'bike_id'),
 (9, 'subscription_type'),
 (10, 'zip_code')]

In [4]:
list(enumerate(stationsHeader.split(",")))

[(0, 'id'),
 (1, 'name'),
 (2, 'lat'),
 (3, 'long'),
 (4, 'dock_count'),
 (5, 'city'),
 (6, 'installation_date')]

In [5]:
stationsIndexed = stations.keyBy(lambda station: station[0])
stationsIndexed.take(2)

[('2',
  ['2',
   'San Jose Diridon Caltrain Station',
   '37.329732',
   '-121.90178200000001',
   '27',
   'San Jose',
   '8/6/2013']),
 ('3',
  ['3',
   'San Jose Civic Center',
   '37.330698',
   '-121.888979',
   '15',
   'San Jose',
   '8/5/2013'])]

In [6]:
tripsByStartTerminals = trips.keyBy(lambda trip: trip[4])
tripsByEndTerminals = trips.keyBy(lambda trip: trip[7])
tripsByStartTerminals.take(1)

[('66',
  ['4576',
   '63',
   '8/29/2013 14:13',
   'South Van Ness at Market',
   '66',
   '8/29/2013 14:14',
   'South Van Ness at Market',
   '66',
   '520',
   'Subscriber',
   '94127'])]

In [7]:
startTrips = stationsIndexed.join(tripsByStartTerminals)
endTrips = stationsIndexed.join(tripsByEndTerminals)
startTrips.take(1)

[('5',
  (['5',
    'Adobe on Almaden',
    '37.331415',
    '-121.8932',
    '19',
    'San Jose',
    '8/5/2013'],
   ['4502',
    '210',
    '8/29/2013 13:28',
    'Adobe on Almaden',
    '5',
    '8/29/2013 13:31',
    'San Jose Civic Center',
    '3',
    '664',
    'Subscriber',
    '95112']))]

In [8]:
from datetime import datetime

def parse_station(row):
    try:
        return {
            'station_id': int(row[0]),
            'name': row[1],
            'lat': float(row[2]),
            'long': float(row[3]),
            'dockcount': int(row[4]),
            'landmark': row[5],
            'installation': datetime.strptime(row[6], '%m/%d/%Y')
        }
    except Exception as e:
        return None

def parse_trip(row):
    try:
        return {
            'trip_id': int(row[0]),
            'duration': int(row[1]),
            'start_date': datetime.strptime(row[2], '%m/%d/%Y %H:%M'),
            'start_station_name': row[3],
            'start_station_id': int(row[4]),
            'end_date': datetime.strptime(row[5], '%m/%d/%Y %H:%M'),
            'end_station_name': row[6],
            'end_station_id': int(row[7]),
            'bike_id': int(row[8]),
            'subscription_type': row[9],
            'zip_code': row[10]
        }
    except Exception as e:
        return None
    
stationsInternal = stations.map(parse_station).filter(lambda x: x is not None)
tripsInternal = trips.map(parse_trip).filter(lambda x: x is not None)
tripsInternal.take(1)[0]

{'trip_id': 4576,
 'duration': 63,
 'start_date': datetime.datetime(2013, 8, 29, 14, 13),
 'start_station_name': 'South Van Ness at Market',
 'start_station_id': 66,
 'end_date': datetime.datetime(2013, 8, 29, 14, 14),
 'end_station_name': 'South Van Ness at Market',
 'end_station_id': 66,
 'bike_id': 520,
 'subscription_type': 'Subscriber',
 'zip_code': '94127'}

## Выполнение заданий

### Задание 1. Найти велосипед с максимальным временем пробега.

In [9]:
bike_id, duration = tripsInternal.map(lambda t: (t['bike_id'], t['duration'])) \
    .reduceByKey(lambda a, b: a + b) \
    .takeOrdered(1, key=lambda x: -x[1])[0]
(bike_id, duration)

(535, 18611693)

In [10]:
amount_trips = tripsInternal.filter(lambda t: t['bike_id'] == bike_id).count()
amount_trips

1328

In [11]:
print(f"Bike {bike_id} travelled {amount_trips} times with total duration = {duration} s ({duration / 3600:.2f} h)")

Bike 535 travelled 1328 times with total duration = 18611693 s (5169.91 h)


### Задание 2. Найти наибольшее геодезическое расстояние между станциями.

In [12]:
stationsInternal.collect()[0]

{'station_id': 2,
 'name': 'San Jose Diridon Caltrain Station',
 'lat': 37.329732,
 'long': -121.90178200000001,
 'dockcount': 27,
 'landmark': 'San Jose',
 'installation': datetime.datetime(2013, 8, 6, 0, 0)}

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

stationsRDD = stationsInternal.map(lambda s: (
    s['station_id'], 
    s['lat'], 
    s['long'], 
    s['name']
))

# сделано через декартово произведение и исключение повторяющихся пар
stations = stationsRDD.cartesian(stationsRDD) \
                   .filter(lambda x: x[0][0] < x[1][0]) \
                   .map(lambda x: (
                            x[0][0], x[0][3],
                            x[1][0], x[1][3],
                            haversine(x[0][1], x[0][2], x[1][1], x[1][2])
                    )).takeOrdered(1, key=lambda x: -x[4])[0]

print(f"Stations with the highest distance: {stations[1]} <--> {stations[3]} | {stations[4]} km")

Stations with the highest distance: SJSU - San Salvador at 9th <--> Embarcadero at Sansome | 69.92087595428183 km


### Задание 3. Найти путь велосипеда с максимальным временем пробега через станции.

In [ ]:
bike_trips = tripsInternal \
             .filter(lambda t: t['bike_id'] == bike_id) \
             .map(lambda t: (t['duration'], t)) \
             .sortByKey(ascending=False) \
             .map(lambda x: x[1])


print(f"Top 10 {bike_id}-bike trips")
print(f"{'#':<5} {'Duration':<15} {'Start':<25} {'Finish':<25}")
print("-" * 70)

for idx, trip in enumerate(bike_trips.take(10), 1):
    duration_min = trip['duration'] / 60
    start_name = trip['start_station_name'][:23] + ".." if len(trip['start_station_name']) > 25 else trip['start_station_name']
    end_name = trip['end_station_name'][:23] + ".." if len(trip['end_station_name']) > 25 else trip['end_station_name']
    print(f"{idx:<3} {duration_min:>8.1f} min    {start_name:<25} - {end_name}")

total_trips = tripsInternal.filter(lambda t: t['bike_id'] == bike_id).count()
unique_stations = tripsInternal \
                    .filter(lambda t: t['bike_id'] == bike_id) \
                    .flatMap(lambda t: [t['start_station_name'], t['end_station_name']]) \
                    .distinct() \
                    .count()

print(f"Total trips with bike {bike_id}: {total_trips}")
print(f"Unique bikes {bike_id}: {unique_stations}")

Top 10 535-bike trips
#     Duration        Start                     Finish                   
----------------------------------------------------------------------
1   287840.0 min    South Van Ness at Market  - 2nd at Folsom
2     1460.6 min    Powell Street BART        - Civic Center BART (7th ..
3      561.0 min    San Francisco Caltrain .. - Steuart at Market
4      431.8 min    Market at 10th            - San Francisco Caltrain ..
5      419.6 min    2nd at Folsom             - Mechanics Plaza (Market..
6      415.3 min    Powell at Post (Union S.. - Powell at Post (Union S..
7      379.8 min    Market at Sansome         - Market at Sansome
8      372.7 min    Broadway St at Battery St - Harry Bridges Plaza (Fe..
9      352.6 min    Grant Avenue at Columbu.. - Grant Avenue at Columbu..
10     316.4 min    Post at Kearney           - Post at Kearney
Total trips with bike 535: 1328
Unique bikes 535: 37


### Задание 4. Найти количество велосипедов в системе.

In [15]:
unique_bikes = tripsInternal.map(lambda t: t['bike_id']).distinct()
bike_count = unique_bikes.count()

print(f"Num of bikes in system: {bike_count:,}")

usage_stats = tripsInternal \
            .map(lambda t: (t['bike_id'], 1)) \
            .reduceByKey(lambda a, b: a + b) \
            .map(lambda x: x[1]).stats()

print(f"Mean of trips: {usage_stats.mean():.1f}")
print(f"Min: {int(usage_stats.min())}, Max: {int(usage_stats.max())}")

Num of bikes in system: 700
Mean of trips: 957.1
Min: 6, Max: 2061


### Задание 5. Найти пользователей потративших на поездки более 3 часов.

In [16]:
user_time = tripsInternal \
            .filter(lambda t: t.get('zip_code')) \
            .map(lambda t: (t.get('zip_code', 'unknown'), t['duration'])) \
            .reduceByKey(lambda a, b: a + b)

THREE_HOURS_SEC = 3 * 3600
heavy_users = user_time.filter(lambda x: x[1] > THREE_HOURS_SEC)

print(f"Users with trip duration > 3 hours")
print(f"  Threshold: {THREE_HOURS_SEC} seconds ({THREE_HOURS_SEC/3600:.0f} hours)")
print(f"  Users: {heavy_users.count()}")

# топ 10 по времени
top_users = heavy_users.takeOrdered(10, key=lambda x: -x[1])
if top_users:
    print(f"\n{'Zip/ID':<12} {'Time (hours)':>15} {'Trips':>10}")
    print("-" * 40)
    for user_id, total_sec in top_users:
        trip_count = tripsInternal.filter(lambda t: t.get('zip_code') == user_id).count()
        print(f"{user_id:<12} {total_sec/3600:>15.2f} {trip_count:>10,}")

Users with trip duration > 3 hours
  Threshold: 10800 seconds (3 hours)
  Users: 3660

Zip/ID          Time (hours)      Trips
----------------------------------------
94107               13821.43     78,704
nil                 12701.54     10,682
94105                7110.04     42,672
94133                6010.47     31,359
94102                5313.34     19,757
94103                5313.16     26,673
95531                4797.33          1
94111                3956.94     21,409
95112                3539.55     11,564
94109                3349.20     13,989
